# 12. 정책 백테스트 — "두 마리 토끼"는 과거 데이터로 어디까지 검증되는가

발표자료 17페이지(기대 효과)는 **제안의 기대치**다. 이 노트북은 보유 데이터로 검증 가능한 부분을 실제로 백테스트한다.

| 항목 | 검증 가능? | 방법 |
|------|-----------|------|
| 🐇 토끼 1 — 차등 임계값이 단일 기준보다 나은가 | ✅ 가능 (승인된 30.7만 명 한정) | LightGBM OOF 위험점수에 두 정책을 소급 적용, 같은 승인 규모에서 결과 비교 |
| 🛡 토끼 2 — 조기 경보 규칙이 디폴트를 미리 잡는가 | ✅ 가능 (백테스트) | 행동 변수 트리거를 고위험군에 소급 적용, 적중률(precision)·포착률(recall) 측정 |
| 현재 거절 상태인 사람에 대한 효과 | ❌ 불가능 | 거절자의 상환 결과가 데이터에 없음 (Reject Inference) → 섀도우 테스트·파일럿 필요 |

**공정성 장치**: 위험 점수는 OOF(Out-Of-Fold) 예측 — 각 사람의 점수는 그 사람을 학습에 쓰지 않은 모델이 매긴 것이라 컨닝이 없다.


In [1]:
import warnings; warnings.filterwarnings('ignore')
import numpy as np, pandas as pd
import os

BASE = '.'
oof = pd.read_csv(os.path.join(BASE, 'ml_oof_predictions.csv'))
lab = pd.read_csv(os.path.join(BASE, 'act2_cluster_labels.csv'))
df = oof.merge(lab[['SK_ID_CURR','only_B','risk_AB']], on='SK_ID_CURR', how='inner')
df['grp'] = np.where(df['risk_AB']==1, '고위험', np.where(df['only_B']==1, '회복형', '일반'))

print(f'표본 {len(df):,}명 · OOF 점수 범위 {df.oof_pred.min():.3f}~{df.oof_pred.max():.3f}')
print(df.groupby('grp').agg(인원=('TARGET','count'), 실제연체율=('TARGET','mean'),
                            평균위험점수=('oof_pred','mean')).round(4).to_string())

표본 307,511명 · OOF 점수 범위 0.005~0.970
         인원   실제연체율  평균위험점수
grp                        
고위험   18621  0.2121  0.6636
일반   260679  0.0727  0.3377
회복형   28211  0.0683  0.3440


## 토끼 1 — 차등 임계값 백테스트

**설계**: 두 정책의 전체 승인 인원을 똑같이 맞춘다(승인율 90%). 차이는 배분뿐이다.
- **단일 정책**: 전원에게 같은 위험점수 컷 적용
- **차등 정책**: 회복형은 컷을 20% 관대하게(0.6/0.5 비율), 고위험은 40% 엄격하게(0.3/0.5 비율), 일반은 전체 인원이 맞도록 조정

같은 승인 규모에서 ① 승인 집단의 실제 연체율, ② 잘못 거절한 우량 고객 수(실제로는 갚은 사람), ③ 교체된 사람들의 실제 성과를 비교한다.


In [2]:
APPROVAL_RATE = 0.90

# ── 단일 정책: 전원 동일 컷 ──
t_single = df['oof_pred'].quantile(APPROVAL_RATE)
df['appr_single'] = df['oof_pred'] < t_single
n_target = int(df['appr_single'].sum())

# ── 차등 정책: 회복형 1.2x 관대 · 고위험 0.6x 엄격 · 일반은 총 승인 인원 맞춤 ──
t_rec, t_high = t_single*1.2, t_single*0.6
m_rec  = (df.grp=='회복형') & (df.oof_pred < t_rec)
m_high = (df.grp=='고위험') & (df.oof_pred < t_high)
need_gen = n_target - int(m_rec.sum()) - int(m_high.sum())
gen_preds = df.loc[df.grp=='일반','oof_pred'].sort_values()
t_gen = gen_preds.iloc[need_gen-1]
df['appr_diff'] = m_rec | m_high | ((df.grp=='일반') & (df.oof_pred <= t_gen))

def policy_report(col, name):
    a = df[df[col]]; r = df[~df[col]]
    print(f'=== {name} ===  승인 {len(a):,}명')
    print(f'  승인 집단 실제 연체율      : {a.TARGET.mean()*100:.3f}%  (연체자 {a.TARGET.sum():,}명 승인)')
    print(f'  잘못 거절한 우량 고객      : {(r.TARGET==0).sum():,}명 (실제로는 갚은 사람)')
    for g in ['회복형','일반','고위험']:
        ga = a[a.grp==g]; gt = df[df.grp==g]
        print(f'    {g:<4} 승인율 {len(ga)/len(gt)*100:5.1f}% · 승인분 연체율 {ga.TARGET.mean()*100:5.2f}%')

policy_report('appr_single', '단일 정책 (전원 같은 컷)')
print()
policy_report('appr_diff',  '차등 정책 (그룹별 컷 조정)')

# ── 교체 효과: 누가 들어오고 누가 나갔나 ──
saved   = df[df.appr_diff & ~df.appr_single]   # 차등 덕분에 추가 승인
blocked = df[~df.appr_diff & df.appr_single]   # 차등 때문에 걸러짐
d_single = df[df.appr_single].TARGET.sum(); d_diff = df[df.appr_diff].TARGET.sum()
verdict = '개선 ✅' if d_diff < d_single else '악화 ⚠️'
print(f'''
=== 같은 승인 규모에서 "교체"된 사람들 ===
🟢 추가 승인 (완화 덕분): {len(saved):,}명 — 실제 연체율 {saved.TARGET.mean()*100:.2f}%
🔴 대신 거절 (강화 때문): {len(blocked):,}명 — 실제 연체율 {blocked.TARGET.mean()*100:.2f}%
→ 승인 집단 연체자: 단일 {d_single:,}명 → 차등 {d_diff:,}명 ({d_diff-d_single:+,}명) = {verdict}''')
if d_diff >= d_single:
    print('''→ 해석: 모델 점수가 이미 거절이력·신용점수 정보를 반영하고 있어서(노트북 09 피처),
  점수 "위에" 그룹별 컷을 다시 얹으면 — 컷 완화로 들어오는 사람은 그 그룹의 최고위험층,
  컷 강화로 나가는 사람은 점수상 무난한 층이라 교체가 손해가 된다. (점수의 충분성)
  ⇒ 시사점: "그룹별 컷 조정"이 아니라, 아래처럼 "점수 밖 자동거절 관행 제거"가 올바른 구현이다.''')

# ── 관점 2: 진짜 제안 — "거절이력 자동거절" 관행 vs 점수 기반 심사 ──
rec = df[df.grp=='회복형']; rec_a = rec[rec.appr_single]
print(f'''
=== 제안의 원형: "거절 이력 자동거절"을 없애면? ===
관행 A (거절이력 = 자동거절)  : 회복형 {len(rec):,}명 전원 탈락 → 실제 상환자 {(rec.TARGET==0).sum():,}명을 잃음
관행 B (점수로만 심사, 단일 컷): 회복형 {len(rec_a):,}명({len(rec_a)/len(rec)*100:.1f}%) 승인 · 승인분 연체율 {rec_a.TARGET.mean()*100:.2f}%
→ 포트폴리오 평균({df[df.appr_single].TARGET.mean()*100:.2f}%)과 비슷한 수준 = 거절이력 딱지를 떼고
  점수로 심사해도 건전성이 유지된다 — "재신청 자동거절 완화" 제안의 실증 근거 ✅''')

=== 단일 정책 (전원 같은 컷) ===  승인 276,759명
  승인 집단 실제 연체율      : 5.694%  (연체자 15,760명 승인)
  잘못 거절한 우량 고객      : 21,687명 (실제로는 갚은 사람)
    회복형  승인율  96.1% · 승인분 연체율  5.99%
    일반   승인율  92.0% · 승인분 연체율  5.42%
    고위험  승인율  52.7% · 승인분 연체율 11.50%

=== 차등 정책 (그룹별 컷 조정) ===  승인 276,759명
  승인 집단 실제 연체율      : 5.958%  (연체자 16,490명 승인)
  잘못 거절한 우량 고객      : 22,417명 (실제로는 갚은 사람)
    회복형  승인율  99.8% · 승인분 연체율  6.77%
    일반   승인율  94.7% · 승인분 연체율  5.87%
    고위험  승인율   9.7% · 승인분 연체율  5.29%

=== 같은 승인 규모에서 "교체"된 사람들 ===
🟢 추가 승인 (완화 덕분): 8,000명 — 실제 연체율 22.04%
🔴 대신 거절 (강화 때문): 8,000명 — 실제 연체율 12.91%
→ 승인 집단 연체자: 단일 15,760명 → 차등 16,490명 (+730명) = 악화 ⚠️
→ 해석: 모델 점수가 이미 거절이력·신용점수 정보를 반영하고 있어서(노트북 09 피처),
  점수 "위에" 그룹별 컷을 다시 얹으면 — 컷 완화로 들어오는 사람은 그 그룹의 최고위험층,
  컷 강화로 나가는 사람은 점수상 무난한 층이라 교체가 손해가 된다. (점수의 충분성)
  ⇒ 시사점: "그룹별 컷 조정"이 아니라, 아래처럼 "점수 밖 자동거절 관행 제거"가 올바른 구현이다.

=== 제안의 원형: "거절 이력 자동거절"을 없애면? ===
관행 A (거절이력 = 자동거절)  : 회복형 28,211명 전원 탈락 → 실제 상환자 26,285명을 잃음
관행 B (점수로만 심사, 단일 컷): 회복형 27,097명(96.1%) 승인 · 승

## 토끼 2 — 조기 경보 백테스트

**설계**: 발표자료의 트리거(휴대폰 최근 변경, 할부 지연, 과거 거절 비율)를 **고위험군 18,621명에 소급 적용**했다면 —
실제 디폴트(21.2%)를 얼마나 미리 플래그할 수 있었는지 측정한다.
- **적중률(precision)**: 경보가 울린 사람 중 실제 연체 비율 (오경보가 적은가)
- **포착률(recall)**: 실제 연체자 중 경보가 미리 울린 비율 (놓치지 않는가)


In [3]:
DATA = os.path.join(BASE, 'home-credit-default-risk')
app  = pd.read_csv(os.path.join(DATA,'application_train.csv'),
                   usecols=['SK_ID_CURR','DAYS_LAST_PHONE_CHANGE','DAYS_ID_PUBLISH'])
prev = pd.read_csv(os.path.join(DATA,'previous_application.csv'),
                   usecols=['SK_ID_CURR','SK_ID_PREV','NAME_CONTRACT_STATUS'])
inst = pd.read_csv(os.path.join(DATA,'installments_payments.csv'),
                   usecols=['SK_ID_CURR','DAYS_INSTALMENT','DAYS_ENTRY_PAYMENT'])

pa = prev.groupby('SK_ID_CURR').agg(total=('SK_ID_PREV','count'),
        refused=('NAME_CONTRACT_STATUS', lambda x:(x=='Refused').sum())).reset_index()
pa['refused_ratio'] = pa['refused']/pa['total']
inst['is_late'] = (inst['DAYS_ENTRY_PAYMENT'] > inst['DAYS_INSTALMENT']).astype(int)
ia = inst.groupby('SK_ID_CURR').agg(late_ratio=('is_late','mean')).reset_index()

bt = df.merge(app, on='SK_ID_CURR', how='left').merge(pa[['SK_ID_CURR','refused_ratio']],
        on='SK_ID_CURR', how='left').merge(ia, on='SK_ID_CURR', how='left')
bt['refused_ratio'] = bt['refused_ratio'].fillna(0); bt['late_ratio'] = bt['late_ratio'].fillna(0)

# 트리거 (발표자료 슬라이드 18 규칙)
bt['trig_phone'] = -bt['DAYS_LAST_PHONE_CHANGE'] < 365          # 1년 내 휴대폰 변경
bt['trig_late']  = bt['late_ratio'] > 0.20                       # 할부 5건 중 1건 이상 지연
bt['trig_ref']   = bt['refused_ratio'] > 0.50                    # 과거 신청 절반 이상 거절
bt['alarm'] = bt[['trig_phone','trig_late','trig_ref']].any(axis=1)

hi = bt[bt.grp=='고위험']
al = hi[hi.alarm]
prec  = al.TARGET.mean()*100
rec   = al.TARGET.sum()/hi.TARGET.sum()*100
print(f'고위험군 {len(hi):,}명 (실제 디폴트 {hi.TARGET.sum():,}명 · 기본 연체율 {hi.TARGET.mean()*100:.1f}%)')
print(f'경보 발령: {len(al):,}명 ({len(al)/len(hi)*100:.1f}%)')
print(f'  적중률(precision): 경보 울린 사람의 실제 연체율 {prec:.1f}%  (기본 대비 {prec/(hi.TARGET.mean()*100):.2f}배)')
print(f'  포착률(recall)   : 실제 연체자 {hi.TARGET.sum():,}명 중 {al.TARGET.sum():,}명({rec:.1f}%)을 사전 플래그')
print()
for c,lbl in [('trig_phone','휴대폰 1년내 변경'),('trig_late','할부 지연율>20%'),('trig_ref','거절비율>50%')]:
    t = hi[hi[c]]
    print(f'  단일 트리거 [{lbl:<12}] 발령 {len(t):,}명 · 적중률 {t.TARGET.mean()*100:.1f}%')

# ── 비교: 같은 발령량을 "모델 점수 상위"로 채우면? ──
k = len(al)
sc = hi.nlargest(k, 'oof_pred')
print(f'''
=== 규칙 경보 vs 점수 경보 (같은 발령량 {k:,}명) ===
규칙 기반 (트리거 3종)   : 적중률 {prec:.1f}% · 포착률 {rec:.1f}%
점수 기반 (위험점수 상위) : 적중률 {sc.TARGET.mean()*100:.1f}% · 포착률 {sc.TARGET.sum()/hi.TARGET.sum()*100:.1f}%''')

고위험군 18,621명 (실제 디폴트 3,949명 · 기본 연체율 21.2%)
경보 발령: 11,103명 (59.6%)
  적중률(precision): 경보 울린 사람의 실제 연체율 22.3%  (기본 대비 1.05배)
  포착률(recall)   : 실제 연체자 3,949명 중 2,478명(62.8%)을 사전 플래그

  단일 트리거 [휴대폰 1년내 변경  ] 발령 5,999명 · 적중률 21.1%
  단일 트리거 [할부 지연율>20%  ] 발령 3,505명 · 적중률 23.9%
  단일 트리거 [거절비율>50%    ] 발령 5,025명 · 적중률 25.0%

=== 규칙 경보 vs 점수 경보 (같은 발령량 11,103명) ===
규칙 기반 (트리거 3종)   : 적중률 22.3% · 포착률 62.8%
점수 기반 (위험점수 상위) : 적중률 29.2% · 포착률 82.0%


## 결론 — 백테스트는 가능했고, 결과는 "구현 방식"을 바꾸라고 말한다

### 🐇 토끼 1 (재심사 완화) — 방향은 검증 ✅, 구현은 수정 필요 ⚠️
- **"그룹별 컷 조정"(회복형 관대 × 고위험 엄격)은 역효과**: 같은 승인 규모에서 승인 연체자가 오히려 증가.
  이유 — 모델 점수가 이미 거절이력·신용점수를 피처로 반영하고 있어(점수의 충분성), 점수 위에 그룹 컷을 얹으면
  들어오는 사람은 그 그룹의 최고위험층, 나가는 사람은 점수상 무난한 층이 된다.
- **"거절이력 자동거절 제거"는 실증됨**: 점수로만 심사하면 회복형의 96%가 승인되고 그 승인분 연체율은
  포트폴리오 평균 수준 — 거절이력 딱지를 근거로 한 일괄 탈락이 손해라는 것이 17페이지 제안의 올바른 형태.

### 🛡 토끼 2 (조기 경보) — 규칙보다 점수 ⚠️
- 행동 트리거 3종(휴대폰·할부지연·거절비율)은 고위험군 안에서는 변별력이 약함(적중률 ≈ 기본 연체율).
  고위험군 정의 자체가 이 행동들과 겹치기 때문.
- 같은 발령량이라면 **모델 위험점수 상위를 경보로 쓰는 쪽이 우월** → 경보 시스템은 규칙이 아니라 점수 기반으로 설계.

### ❌ 여전히 검증 불가능한 것 (데이터 구조상)
- 이 백테스트는 **승인됐던 사람들** 위에서의 시뮬레이션이다. 현재 거절 상태인 재신청자의 실제 상환 성과는
  데이터에 없다(Reject Inference) → **섀도우 테스트 + 소규모 파일럿**이 여전히 필요한 이유.

**한 줄 요약**: "두 마리 토끼"는 과거 데이터로 백테스트할 수 있었고, 그 결과는 제안의 *방향*(자동거절 완화 + 사전 감지)을
지지하되 *구현*(그룹별 컷 ✕ → 점수 중심 ○, 규칙 경보 ✕ → 점수 경보 ○)을 교정해 줬다. 백테스트를 미리 돌려본 것 자체가
"바로 시행하지 않고 검증부터 한다"는 실행 계획의 가치를 증명한다.
